# Lesson 02 - Microsoft Agent Frameworkの探求

**Microsoft Agent Framework (MAF)** はAIエージェント構築のための統一フレームワークです。4つのコアビルディングブロックからなる、シンプルで組み合わせ可能なアーキテクチャを提供します：

- **Client** – AIモデルのエンドポイントに接続し、通信を処理する
- **Agent** – クライアントを命令やツール定義でラップする
- **Tools** – モデルが呼び出せるカスタム関数でエージェントの機能を拡張する
- **Session** – マルチターンの対話に向けた会話履歴を保持する

このレッスンでは、これらの概念を使って、目的地の空き状況を確認する**旅行予約エージェント**を構築します。


## セットアップ


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## エージェントフレームワークのアーキテクチャの理解

Microsoft Agent Frameworkはレイヤードアーキテクチャに従っています：

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **クライアント** – `AzureAIProjectAgentProvider`はAzure OpenAIのデプロイメントに接続します。認証、リクエストのフォーマット、レスポンスの解析を処理します。
2. **エージェント** – クライアントから`provider.create_agent()`で作成され、モデルアクセスと指示（システムプロンプト）およびツールを組み合わせます。
3. **ツール** – `@tool`でデコレートされたPython関数で、エージェントが操作を実行したりデータを取得したりするために呼び出すことができます。
4. **セッション** – 会話履歴を保存する`AgentSession`オブジェクト（`agent.create_session()`で作成）で、エージェントがこれまでのコンテキストを記憶し複数回の対話が可能になります。

それぞれのレイヤーを順を追って構築していきましょう。


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @toolデコレーターを使ったツールの追加

ツールはエージェントがテキスト生成以外のアクションを実行できるようにします。`@tool`デコレーターは通常のPython関数をエージェントが呼び出せるものに変換します。

主なポイント：
- モデルが各パラメータを理解できるように`Annotated[type, "説明"]`を使います。
- ドキュメンテーション文字列がモデルに表示されるツールの説明になります。
- `approval_mode="never_require"`はユーザー確認なしでツールが自動的に実行されることを意味します。


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ツールを使ったエージェントの作成

ここでクライアント、指示、およびツールを組み合わせてエージェントを作成します。`instructions` はシステムプロンプトとして機能し、エージェントの人格や振る舞いを定義します。


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## セッションを使ったマルチターン会話

`AgentSession`（`agent.create_session()`で作成）は、会話内のすべてのメッセージを追跡します。同じセッションを各`agent.run()`呼び出しに渡すことで、エージェントは会話の全履歴にアクセスでき、以前のメッセージを参照できます。

`tools=[check_destination_availability]`を渡すことで、エージェントは各ターンで利用可能性チェッカーを呼び出すことができます。


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## まとめ

このレッスンでは、Microsoft Agent Framework の4つの柱について学びました。

| 概念 | 学んだこと |
|---------|------------------|
| **クライアント** | `AzureAIProjectAgentProvider` は資格情報ベースの認証で Azure OpenAI に接続します |
| **エージェント** | `provider.create_agent()` はモデル接続、指示、名前をひとまとめにします |
| **ツール** | `@tool` デコレーターはエージェントが呼び出せるPython関数を公開します |
| **セッション** | `agent.create_session()` は複数ターンにわたる会話履歴を保持します |

これらのビルディングブロックは組み合わさり、自然な会話ができ、外部関数を呼び出し、コンテキストを維持できるエージェントを作成します。これが後のレッスンでのより高度なエージェントパターンの基礎となります。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：  
本書類はAI翻訳サービス「Co-op Translator」（https://github.com/Azure/co-op-translator）を使用して翻訳されています。正確性の向上に努めておりますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご了承ください。原文の言語による文書が正式な情報源として扱われるべきです。重要な情報については、専門の人間による翻訳を推奨いたします。本翻訳の利用により生じた誤解や解釈の違いについて、一切責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
